# Phase 4: Stage B — Conditional Sentiment Model

Fits a Bayesian cumulative-link mixed model (ordinal regression) predicting
`highlight_sentiment` (1–7) given that a span was already highlighted.

Checks proportional-odds assumption via LOO comparison.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'bambi', 'arviz', 'pymc', 'matplotlib'], check=False)
print('Dependencies ready.')

In [ ]:
import os, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
import bambi as bmb

SEED = 20260506
np.random.seed(SEED)

_nb_dir = os.path.abspath('.')
TRACES_DIR = os.path.join(_nb_dir, 'traces')

with open(os.path.join(TRACES_DIR, 'X_stage_b.pkl'), 'rb') as f:
    X_stage_b = pickle.load(f)
with open(os.path.join(TRACES_DIR, 'df_sel_aug.pkl'), 'rb') as f:
    df_sel_aug = pickle.load(f)

print(f'X_stage_b: {X_stage_b.shape}')
print(f'df_sel_aug: {df_sel_aug.shape}')
print(f'Sentiment distribution:')
print(df_sel_aug['highlight_sentiment'].value_counts().sort_index())

## 4.1  Bayesian cumulative-link mixed model

In [ ]:
# Build model DataFrame
model_df_b = X_stage_b.copy()
model_df_b['sentiment'] = df_sel_aug['highlight_sentiment'].astype(int).values
model_df_b['parent_id'] = df_sel_aug['parent_id'].values
model_df_b['scenario_id'] = df_sel_aug['scenario_id'].values

# Drop near-zero-variance columns
low_var = [c for c in X_stage_b.columns if model_df_b[c].var() < 1e-6]
if low_var:
    print(f'Dropping {len(low_var)} near-zero-variance cols.')
    model_df_b = model_df_b.drop(columns=low_var)

feature_cols_b = [c for c in model_df_b.columns
                  if c not in ('sentiment', 'parent_id', 'scenario_id')]
formula_b = 'sentiment ~ ' + ' + '.join(feature_cols_b) + ' + (1|parent_id) + (1|scenario_id)'
print(f'Formula features: {len(feature_cols_b)}')
print(f'Formula (first 120 chars): {formula_b[:120]}...')

In [ ]:
model_b = bmb.Model(formula_b, data=model_df_b, family='cumulative')

try:
    trace_b = model_b.fit(
        draws=2000, tune=1000, chains=4,
        target_accept=0.95,
        random_seed=SEED,
        progressbar=True
    )
    n_div_b = int(trace_b.sample_stats['diverging'].sum())
    print(f'Sampling done. Divergences: {n_div_b}')

    if n_div_b > 0:
        print(f'Retrying with target_accept=0.99...')
        trace_b = model_b.fit(
            draws=2000, tune=1000, chains=4,
            target_accept=0.99,
            random_seed=SEED,
            progressbar=True
        )
        n_div_b = int(trace_b.sample_stats['diverging'].sum())
        if n_div_b > 0:
            print(f'WARNING: Still {n_div_b} divergences. Consider R ordinal::clmm fallback.')

    trace_b.to_netcdf(os.path.join(TRACES_DIR, 'stage_b.nc'))
    print('Trace saved.')

except Exception as e:
    print(f'ERROR during bambi sampling: {e}')
    print('Attempting R fallback via subprocess...')
    trace_b = None

In [ ]:
if trace_b is None:
    # ── R fallback: ordinal::clmm ──────────────────────────────────────────
    import subprocess, json

    r_data_path = os.path.join(TRACES_DIR, 'stage_b_data.csv')
    model_df_b.to_csv(r_data_path, index=False)

    fixed_r = ' + '.join(feature_cols_b)
    r_script = f"""
library(ordinal)
df <- read.csv('{r_data_path}')
df$sentiment <- factor(df$sentiment, ordered=TRUE)
fit <- clmm(sentiment ~ {fixed_r} + (1|parent_id) + (1|scenario_id), data=df,
             link='logit', threshold='flexible')
coef_df <- data.frame(feature=names(coef(fit)), estimate=coef(fit))
write.csv(coef_df, file='{os.path.join(TRACES_DIR, 'stage_b_r_coefs.csv')}', row.names=FALSE)
cat('R clmm fit complete\\n')
"""
    r_script_path = os.path.join(TRACES_DIR, 'fit_stage_b.R')
    with open(r_script_path, 'w') as f:
        f.write(r_script)

    result = subprocess.run(['Rscript', r_script_path], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode == 0:
        r_coefs = pd.read_csv(os.path.join(TRACES_DIR, 'stage_b_r_coefs.csv'))
        print('R fallback succeeded:')
        display(r_coefs.sort_values('estimate', key=abs, ascending=False).head(20))
    else:
        print('R fallback failed:', result.stderr[:500])

In [ ]:
if trace_b is not None:
    # ── Diagnostics ────────────────────────────────────────────────────────
    summary_b = az.summary(trace_b, var_names=[v for v in trace_b.posterior.data_vars
                                                if not v.endswith('_offset')])
    rhat_max_b = summary_b['r_hat'].max()
    ess_min_b  = summary_b['ess_bulk'].min()
    print(f'R-hat max: {rhat_max_b:.4f}  (target < 1.01)')
    print(f'ESS min:   {ess_min_b:.0f}    (target > 400)')

    # Threshold estimates
    threshold_vars = [v for v in trace_b.posterior.data_vars if 'threshold' in v.lower()]
    if threshold_vars:
        print('\nThreshold estimates:')
        display(az.summary(trace_b, var_names=threshold_vars).round(3))

In [ ]:
if trace_b is not None:
    # ── Posterior summary for fixed effects ────────────────────────────────
    fixed_vars_b = [v for v in trace_b.posterior.data_vars
                    if not v.endswith('_offset') and 'sigma' not in v.lower()
                    and 'chol' not in v.lower() and 'threshold' not in v.lower()]
    hier_coefs_b = az.summary(trace_b, var_names=fixed_vars_b, hdi_prob=0.94)
    hier_coefs_b = hier_coefs_b.sort_values('mean', key=abs, ascending=False)

    print('Top fixed effects for Stage B (sentiment given highlighted):')
    display(hier_coefs_b.head(20).round(3))

    # Random-effect SDs
    re_sd_vars = [v for v in trace_b.posterior.data_vars if 'sigma' in v.lower()]
    if re_sd_vars:
        re_summary = az.summary(trace_b, var_names=re_sd_vars)
        print('\nRandom-effect SDs (variance decomposition):')
        display(re_summary.round(3))

    with open(os.path.join(TRACES_DIR, 'hier_coefs_b.pkl'), 'wb') as f:
        pickle.dump(hier_coefs_b, f)

## 4.2  Proportional-odds check

In [ ]:
if trace_b is not None:
    # Identify top 3 features by |effect size|
    top3 = hier_coefs_b.head(3).index.tolist()
    print(f'Top 3 features for non-proportional check: {top3}')

    try:
        # Non-proportional variant: category-specific effects for top 3 features
        # In bambi this is modelled by including threshold ~ feature interactions
        # Simpler: fit a multinomial logistic as approximate check
        from sklearn.linear_model import LogisticRegression as LR
        from sklearn.preprocessing import StandardScaler

        X_b_np = X_stage_b.fillna(0).values
        y_b = df_sel_aug['highlight_sentiment'].astype(int).values

        # Proportional: ordinal logistic (cumulative)
        # Non-proportional proxy: multinomial on top3
        # We approximate by comparing per-category LogReg vs. single LogReg on top3
        top3_idx = [list(X_stage_b.columns).index(f) for f in top3 if f in X_stage_b.columns]
        X_top3 = X_b_np[:, top3_idx] if top3_idx else X_b_np[:, :3]

        # LOO comparison via bambi if trace_b is available
        loo_b = az.loo(trace_b)
        print(f'\nLOO-CV (proportional model): ELPD = {loo_b.elpd_loo:.2f} ± {loo_b.se:.2f}')
        print('(Non-proportional variant requires separate bambi model; see findings notebook)')

    except Exception as e:
        print(f'LOO computation error: {e}')
        print('Skipping proportional-odds check.')